# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarahnjunge/starter-notebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:

!git clone https://github.com/Sarahnjunge/starter-notebooks.git
%cd starter-notebooks
!ls data/raw


Cloning into 'starter-notebooks'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 168 (delta 72), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.89 MiB | 10.03 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/starter-notebooks
content_refresh_anonymized.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


### My rule

I want the review queue to prioritize pages that have meaningful historical visibility but show signs that they may need attention.

A page receives a higher action score when it has substantial historical impressions, is old or has not been updated recently, and has weaker engagement or ranking signals. The rule is intentionally simple and uses fixed conditions rather than fitted model weights.

### Reason codes

* `stale_visible` — the page has meaningful historical visibility and has not been updated recently.
* `low_ctr_visible` — the page receives meaningful impressions but has a relatively low click-through rate.
* `weak_engagement` — the page has meaningful traffic but relatively weak engagement.
* `multiple_signals` — more than one review signal is present.
* `no_clear_signal` — none of the review conditions are triggered.


In [3]:

# Baseline rule signals

visible_signal = df["impressions_90d"] > 0

stale_signal = df["days_since_last_update"] > 180

low_ctr_signal = (
    visible_signal &
    (df["ctr"] < df["ctr"].median())
)

weak_engagement_signal = (
    (df["sessions_90d"] > 0) &
    (df["engagement_rate"] < df["engagement_rate"].median())
)

df["reason_code"] = np.select(
    [
        stale_signal & low_ctr_signal,
        stale_signal,
        low_ctr_signal,
        weak_engagement_signal
    ],
    [
        "multiple_signals",
        "stale_visible",
        "low_ctr_visible",
        "weak_engagement"
    ],
    default="no_clear_signal"
)

# Simple action score: number of independent warning signals
df["action_score"] = (
    stale_signal.astype(int)
    + low_ctr_signal.astype(int)
    + weak_engagement_signal.astype(int)
)

print("Action score distribution:")
print(df["action_score"].value_counts().sort_index())

print("\nReason codes:")
print(df["reason_code"].value_counts())

Action score distribution:
action_score
0    15136
1    14744
2      120
Name: count, dtype: int64

Reason codes:
reason_code
no_clear_signal     15136
low_ctr_visible     14690
multiple_signals      120
stale_visible          54
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)



I rank every page by its baseline action score, with higher scores appearing first. Pages with the same score are ordered using lower CTR as a secondary signal so that the queue remains deterministic.

The final queue contains the page identifier, client identifier, action score, reason code, and the main signals used to explain the ranking. The ranked queue is saved as `work/outputs/baseline_action_score.csv`.


In [4]:
# Build the ranked review queue

ranked_queue = df[
    [
        "content_id",
        "client_id",
        "action_score",
        "reason_code",
        "ctr",
        "impressions_90d",
        "engagement_rate",
        "days_since_last_update"
    ]
].copy()

# Higher action score first, then lower CTR first
ranked_queue = ranked_queue.sort_values(
    by=["action_score", "ctr"],
    ascending=[False, True]
).reset_index(drop=True)

# Add review rank
ranked_queue.insert(0, "rank", ranked_queue.index + 1)

# Create output directory
import os
os.makedirs("work/outputs", exist_ok=True)

# Save ranked queue
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)

print("Rows ranked:", len(ranked_queue))
print("Output written:", output_path)

print("\nTop 20:")
print(ranked_queue.head(20).to_string(index=False))


Rows ranked: 30000
Output written: work/outputs/baseline_action_score.csv

Top 20:
 rank           content_id         client_id  action_score      reason_code  ctr  impressions_90d  engagement_rate  days_since_last_update
    1 content_bfa3d6688324 client_d029fa3a95             2 multiple_signals  0.0               27              0.0                     183
    2 content_b16bd7307b39 client_7f2253d7e2             2 multiple_signals  0.0             4590              0.0                     194
    3 content_a98703986e70 client_d4735e3a26             2 multiple_signals  0.0                1              0.0                     211
    4 content_ab27c30d81f4 client_4ec9599fc2             2 multiple_signals  0.0              103              0.0                     304
    5 content_4f241bad48a3 client_6208ef0f77             2 multiple_signals  0.0              285              0.0                     236
    6 content_1af4aceb4525 client_d4735e3a26             2 multiple_signals  0.0   

## 3. Top-20 review


I review the highest-ranked 20 pages rather than assuming that a high score means the page definitely needs action.

For each page I record the recommended action, the reason code, a confidence note, and what could make the recommendation wrong.

The baseline is a decision-support queue, not proof that a page is declining or that updating it will improve performance.


In [5]:
# Prepare the top-20 pages for manual review

top20 = ranked_queue.head(20).copy()

top20["action"] = np.where(
    top20["action_score"] >= 2,
    "review",
    "monitor"
)

top20["confidence_note"] = np.where(
    top20["impressions_90d"] >= 100,
    "More evidence: meaningful historical visibility",
    "Lower confidence: limited historical visibility"
)

top20["what_could_make_it_wrong"] = np.where(
    top20["impressions_90d"] < 100,
    "Very low impression volume may make the signal unreliable",
    "Historical signals may not reflect current page conditions"
)

review_columns = [
    "rank",
    "content_id",
    "client_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_could_make_it_wrong"
]

print(top20[review_columns].to_string(index=False))

 rank           content_id         client_id action      reason_code                                 confidence_note                                   what_could_make_it_wrong
    1 content_bfa3d6688324 client_d029fa3a95 review multiple_signals Lower confidence: limited historical visibility  Very low impression volume may make the signal unreliable
    2 content_b16bd7307b39 client_7f2253d7e2 review multiple_signals More evidence: meaningful historical visibility Historical signals may not reflect current page conditions
    3 content_a98703986e70 client_d4735e3a26 review multiple_signals Lower confidence: limited historical visibility  Very low impression volume may make the signal unreliable
    4 content_ab27c30d81f4 client_4ec9599fc2 review multiple_signals More evidence: meaningful historical visibility Historical signals may not reflect current page conditions
    5 content_4f241bad48a3 client_6208ef0f77 review multiple_signals More evidence: meaningful historical visibility His

## 4. Weak picks + leakage check


Several top-ranked pages are weak picks because they have very low historical impression volume. A zero CTR is less informative when a page has received only a handful of impressions, so these pages should be treated as lower-confidence review candidates.

The baseline does not use `trend_direction` or `trend_pct`, so the target is not directly leaked into the score. It also does not use `provider_used` or `model_used`, which were excluded because they represent system/product information rather than page-level signals.

The baseline uses only the historical fields available in the starter dataset. However, because the starter dataset has no calendar-date column, I cannot independently prove the exact prediction-time ordering of every field. This is therefore a limitation of the exercise rather than evidence of production-time availability.


In [6]:
# Identify weak picks and check for leakage

weak_picks = top20[top20["impressions_90d"] < 100].copy()

print("Weak top-20 picks (<100 impressions):", len(weak_picks))

print("\nWeak picks:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_code",
            "ctr",
            "impressions_90d"
        ]
    ].to_string(index=False)
)

# Confirm excluded leakage/product fields are not in the ranked queue
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used"
]

print("\nLeakage/product fields present in ranked queue:")
print([
    col for col in leakage_fields
    if col in ranked_queue.columns
])

print("\nFinal ranked queue shape:", ranked_queue.shape)


Weak top-20 picks (<100 impressions): 14

Weak picks:
 rank           content_id  action_score      reason_code  ctr  impressions_90d
    1 content_bfa3d6688324             2 multiple_signals  0.0               27
    3 content_a98703986e70             2 multiple_signals  0.0                1
    6 content_1af4aceb4525             2 multiple_signals  0.0                1
    7 content_74961b456728             2 multiple_signals  0.0                1
    8 content_bbca724138f2             2 multiple_signals  0.0               75
    9 content_0edf498ae135             2 multiple_signals  0.0                3
   10 content_1d10143d4e52             2 multiple_signals  0.0               20
   11 content_36e7b91747fa             2 multiple_signals  0.0                1
   12 content_a34d943a132c             2 multiple_signals  0.0               35
   13 content_7b4e68b406b8             2 multiple_signals  0.0                4
   15 content_2e2a634851a0             2 multiple_signals  0.0    

In [7]:
# Evaluate the baseline ranking against the existing target
# The target is used only for evaluation, not for creating the ranking.

evaluation = ranked_queue.merge(
    df[["content_id", "trend_direction"]],
    on="content_id",
    how="left"
)

# Define the positive class for this starter-data baseline
# "down" represents pages that need attention.
evaluation["is_positive"] = (
    evaluation["trend_direction"] == "down"
).astype(int)

precision_at_20 = evaluation.head(20)["is_positive"].mean()
precision_at_50 = evaluation.head(50)["is_positive"].mean()

print(f"Precision@20: {precision_at_20:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")

print("\nPositive pages in top 20:",
      evaluation.head(20)["is_positive"].sum())

print("Positive pages in top 50:",
      evaluation.head(50)["is_positive"].sum())

Precision@20: 0.500
Precision@50: 0.460

Positive pages in top 20: 10
Positive pages in top 50: 23


In [8]:
# Compare baseline precision with the overall positive-class base rate

base_rate = evaluation["is_positive"].mean()

print(f"Base rate: {base_rate:.3f}")
print(f"Precision@20: {precision_at_20:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")

print("\nBaseline lift:")
print(f"Precision@20 lift: {precision_at_20 / base_rate:.2f}x")
print(f"Precision@50 lift: {precision_at_50 / base_rate:.2f}x")

Base rate: 0.542
Precision@20: 0.500
Precision@50: 0.460

Baseline lift:
Precision@20 lift: 0.92x
Precision@50 lift: 0.85x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.